In [1]:
from src.ingestion import import_data, load_retail_data
import pandas as pd

In [2]:
data = load_retail_data("data/raw")

for name, df in data.items():
    print(f"{name}: {df.shape}")

stores: (54, 5)
train: (3000888, 6)
oil: (1218, 2)
holidays_events: (350, 6)
transactions: (83488, 3)
test: (28512, 5)


In [3]:
for name, df in data.items():
    print(f"\n--- {name} ---")
    print(df.isnull().sum())


--- stores ---
store_nbr    0
city         0
state        0
type         0
cluster      0
dtype: int64

--- train ---
id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64

--- oil ---
date           0
dcoilwtico    43
dtype: int64

--- holidays_events ---
date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64

--- transactions ---
date            0
store_nbr       0
transactions    0
dtype: int64

--- test ---
id             0
date           0
store_nbr      0
family         0
onpromotion    0
dtype: int64


In [4]:
oil = data["oil"]

missing_oil = oil[oil["dcoilwtico"].isnull()]

missing_oil

,date,dcoilwtico
0,2013-01-01,NaN
14,2013-01-21,NaN
34,2013-02-18,NaN
63,2013-03-29,NaN
104,2013-05-27,NaN
132,2013-07-04,NaN
174,2013-09-02,NaN
237,2013-11-28,NaN
256,2013-12-25,NaN
261,2014-01-01,NaN


In [5]:
for date in missing_oil["date"].head(10):
    print(oil.loc[
        (oil["date"] >= date) &
        (oil["date"] <= date),
        ["date", "dcoilwtico"]
    ])

         date  dcoilwtico
0  2013-01-01         NaN
          date  dcoilwtico
14  2013-01-21         NaN
          date  dcoilwtico
34  2013-02-18         NaN
          date  dcoilwtico
63  2013-03-29         NaN
           date  dcoilwtico
104  2013-05-27         NaN
           date  dcoilwtico
132  2013-07-04         NaN
           date  dcoilwtico
174  2013-09-02         NaN
           date  dcoilwtico
237  2013-11-28         NaN
           date  dcoilwtico
256  2013-12-25         NaN
           date  dcoilwtico
261  2014-01-01         NaN


In [6]:
oil.dtypes

date              str
dcoilwtico    float64
dtype: object

In [7]:
oil["date"] = pd.to_datetime(oil["date"])

In [8]:
oil.dtypes

date          datetime64[us]
dcoilwtico           float64
dtype: object

In [9]:
print("First missing date:", missing_oil["date"].min())
print("Last missing date:", missing_oil["date"].max())
print("Total missing prices:", missing_oil["dcoilwtico"].isna().sum())

First missing date: 2013-01-01
Last missing date: 2017-07-04
Total missing prices: 43


In [10]:
for name, df in data.items():
    print(f"\n{name}:")
    print(df.columns.tolist())


stores:
['store_nbr', 'city', 'state', 'type', 'cluster']

train:
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion']

oil:
['date', 'dcoilwtico']

holidays_events:
['date', 'type', 'locale', 'locale_name', 'description', 'transferred']

transactions:
['date', 'store_nbr', 'transactions']

test:
['id', 'date', 'store_nbr', 'family', 'onpromotion']


In [35]:
from src.validations import validate_schema, validate_missing_values, validate_duplicates

In [12]:
results = validate_schema(data)

for name, result in results.items():
    print(f"{name}: {result['status']} — {result['message']}")

stores: success — Schema validation passed
train: success — Schema validation passed
oil: success — Schema validation passed
holidays_events: success — Schema validation passed
transactions: success — Schema validation passed
test: success — Schema validation passed


In [13]:
results = validate_missing_values(data)

for name , result in results.items():
    print(f"{name}: {result['status']} — {result['message']}")

stores: success — No missing values
train: success — No missing values
oil: error — Missing values found: 43
holidays_events: success — No missing values
transactions: success — No missing values
test: success — No missing values


In [14]:
results = validate_duplicates(data)

for name , result in results.items():
    print(f"{name}: {result['status']} — {result['message']}")

stores: success — No duplicate rows
train: success — No duplicate rows
oil: success — No duplicate rows
holidays_events: success — No duplicate rows
transactions: success — No duplicate rows
test: success — No duplicate rows


In [22]:
print(data["stores"]["store_nbr"].duplicated().sum())

0


In [23]:
print(data["train"].duplicated(
    subset=["date", "store_nbr", "family"]
).sum())

0


In [24]:
print(data["transactions"].duplicated(
    subset=["date", "store_nbr"]
).sum())

0


In [25]:
print(data["oil"].duplicated(
    subset=["date"]
).sum())

0


In [26]:
print(data["test"].duplicated(
    subset=["date", "store_nbr", "family"]
).sum())

0


In [27]:
train_stores = set(data["train"]["store_nbr"])
store_master = set(data["stores"]["store_nbr"])

missing_stores = train_stores - store_master

print("Stores referenced in train:", len(train_stores))
print("Stores in stores table:", len(store_master))
print("Missing store references:", missing_stores)

Stores referenced in train: 54
Stores in stores table: 54
Missing store references: set()


In [28]:
transaction_stores = set(data["transactions"]["store_nbr"])
store_master = set(data["stores"]["store_nbr"])

missing_transaction_stores = transaction_stores - store_master

print("Stores referenced in transactions:", len(transaction_stores))
print("Stores in stores table:", len(store_master))
print("Missing store references:", missing_transaction_stores)

Stores referenced in transactions: 54
Stores in stores table: 54
Missing store references: set()


In [29]:
for name in ["train", "test", "oil", "transactions", "holidays_events"]:
    df = data[name]
    print(
        f"{name}: "
        f"{df['date'].min()} → {df['date'].max()}"
    )

train: 2013-01-01 → 2017-08-15
test: 2017-08-16 → 2017-08-31
oil: 2013-01-01 00:00:00 → 2017-08-31 00:00:00
transactions: 2013-01-01 → 2017-08-15
holidays_events: 2012-03-02 → 2017-12-26


In [30]:
oil_dates = pd.date_range(
    start=data["oil"]["date"].min(),
    end=data["oil"]["date"].max(),
    freq="D"
)

missing_oil_dates = oil_dates.difference(data["oil"]["date"])

print("Expected dates:", len(oil_dates))
print("Actual dates:", data["oil"]["date"].nunique())
print("Missing dates:", len(missing_oil_dates))
print(missing_oil_dates)

Expected dates: 1704
Actual dates: 1218
Missing dates: 486
DatetimeIndex(['2013-01-05', '2013-01-06', '2013-01-12', '2013-01-13',
               '2013-01-19', '2013-01-20', '2013-01-26', '2013-01-27',
               '2013-02-02', '2013-02-03',
               ...
               '2017-07-29', '2017-07-30', '2017-08-05', '2017-08-06',
               '2017-08-12', '2017-08-13', '2017-08-19', '2017-08-20',
               '2017-08-26', '2017-08-27'],
              dtype='datetime64[us]', length=486, freq=None)


In [31]:
missing_dates = pd.DataFrame({
    "date": missing_oil_dates
})

missing_dates["day"] = missing_dates["date"].dt.day_name()

print(missing_dates["day"].value_counts())

day
Saturday    243
Sunday      243
Name: count, dtype: int64


In [32]:
oil_missing = data["oil"][data["oil"]["dcoilwtico"].isna()]

print("Missing oil price dates:")
print(oil_missing["date"].tolist())

print("\nMissing values:", len(oil_missing))

Missing oil price dates:
[Timestamp('2013-01-01 00:00:00'), Timestamp('2013-01-21 00:00:00'), Timestamp('2013-02-18 00:00:00'), Timestamp('2013-03-29 00:00:00'), Timestamp('2013-05-27 00:00:00'), Timestamp('2013-07-04 00:00:00'), Timestamp('2013-09-02 00:00:00'), Timestamp('2013-11-28 00:00:00'), Timestamp('2013-12-25 00:00:00'), Timestamp('2014-01-01 00:00:00'), Timestamp('2014-01-20 00:00:00'), Timestamp('2014-02-17 00:00:00'), Timestamp('2014-04-18 00:00:00'), Timestamp('2014-05-26 00:00:00'), Timestamp('2014-07-04 00:00:00'), Timestamp('2014-09-01 00:00:00'), Timestamp('2014-11-27 00:00:00'), Timestamp('2014-12-25 00:00:00'), Timestamp('2015-01-01 00:00:00'), Timestamp('2015-01-19 00:00:00'), Timestamp('2015-02-16 00:00:00'), Timestamp('2015-04-03 00:00:00'), Timestamp('2015-05-25 00:00:00'), Timestamp('2015-07-03 00:00:00'), Timestamp('2015-09-07 00:00:00'), Timestamp('2015-11-26 00:00:00'), Timestamp('2015-12-25 00:00:00'), Timestamp('2016-01-01 00:00:00'), Timestamp('2016-01-18 

In [33]:
train_dates = set(data["train"]["date"])
transaction_dates = set(data["transactions"]["date"])

missing_transaction_dates = train_dates - transaction_dates

print("Dates in train:", len(train_dates))
print("Dates in transactions:", len(transaction_dates))
print("Train dates missing from transactions:", len(missing_transaction_dates))
print(missing_transaction_dates)

Dates in train: 1684
Dates in transactions: 1682
Train dates missing from transactions: 2
{'2016-01-01', '2016-01-03'}


In [34]:
for date in ["2016-01-01", "2016-01-03"]:
    print(f"\n{date}")
    print("Train rows:", (data["train"]["date"] == date).sum())
    print("Transactions rows:", (data["transactions"]["date"] == date).sum())


2016-01-01
Train rows: 1782
Transactions rows: 0

2016-01-03
Train rows: 1782
Transactions rows: 0
